Use this jupyter Notebook to try out code locally.

In [3]:
def fastf1_to_sql(years, track_list, team_list):
    '''Lädt die DB von fastf1 und speichert sie in SQLite-DB, damit schnellere Abfragen möglich sind.'''

    # Verzeichnis für Cache erstellen, damit die Daten lokal gespeichert werden und es somit schneller geht
    if not os.path.exists('f1_cache'):
        os.makedirs('f1_cache')
    fastf1.Cache.enable_cache('f1_cache')

    # existierende db löschen, falls sie existiert
    if os.path.exists('f1_project.db'):
        os.remove('f1_project.db')
        print(f"Existierende Datenbank wurde erfolgreich gelöscht. Neue wird erstellt. Dies kann einige Minuten dauern...")

    conn = sqlite3.connect('f1_project.db')
    
    for year in years:
        schedule = fastf1.get_event_schedule(year)
        # Filtern der Rennstrecken, Benutzung von str.contains für flexible Suche
        filtered_events = schedule[schedule['EventName'].str.contains('|'.join(track_list), case=False)]
        
        for _, event in filtered_events.iterrows():
            try:
                print(f"Lade {year} {event['EventName']}...")
                session = fastf1.get_session(year, event['RoundNumber'], 'R')
                
                # Laden der Runden- und Wetterdaten und Messages, keine Telemetrie für schnellere Verarbeitung
                session.load(laps=True, telemetry=False, weather=True, messages=True)
                
                laps = session.laps.pick_teams(team_list).copy()
                if laps.empty: continue

                weather = session.weather_data
                weather['Time'] = weather['Time'].dt.total_seconds() # Wir runden die Zeitstempel, um sie mit den Laps zu matchen
                
                # Hilfsfunktion: Gab es Regen während dieser Runde?
                def check_rain(lap_row):
                    # Wir schauen, ob im Zeitfenster der Runde Rainfall registriert wurde
                    start = lap_row['Time'].total_seconds() - lap_row['LapTime'].total_seconds()
                    end = lap_row['Time'].total_seconds()
                    rain_in_lap = weather[(weather['Time'] >= start) & (weather['Time'] <= end)]['Rainfall']
                    return 1 if rain_in_lap.any() else 0

                # Zeit in Sekunden umwandeln
                laps['LapTimeSec'] = laps['LapTime'].dt.total_seconds()
                
                # Nötige Spalten für ML Modell
                laps['IsRaining'] = laps.apply(check_rain, axis=1) # Pro Runde prüfen
                laps['AirTemp'] = weather['AirTemp'].mean() # Durchschnitt für die Session
                laps['Track'] = event['EventName']
                laps['Year'] = year
                laps['IsOutlap'] = laps['PitOutTime'].notnull().astype(int)
                laps['IsPitstop'] = laps['PitInTime'].notnull().astype(int)

                # Bereinigen und Speichern
                db_data = laps.dropna(subset=['LapTimeSec'])[[
                    'Year', 'Track', 'Team', 'LapNumber', 'LapTimeSec', 
                    'Compound', 'TyreLife', 'AirTemp', 'IsRaining', 'IsOutlap', 'IsPitstop'
                ]]

                db_data.to_sql('laptimes', conn, if_exists='append', index=False)
                print(f"Erfolg: {year} {event['EventName']}")

            except Exception:
                print(f"Skip {year} {event['EventName']}")
    
    conn.close()

In [6]:
years = range(2010, 2026)
track_list = ['Monaco', 'British']
team_list = ['Ferrari', 'Mercedes']
fastf1_to_sql(years, track_list, team_list)

Existierende Datenbank wurde erfolgreich gelöscht. Neue wird erstellt. Dies kann einige Minuten dauern...
Lade 2010 Monaco Grand Prix...


core           INFO 	Loading data for Monaco Grand Prix - Race [v3.8.1]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
logger      WARNING 	Failed to load session info data!
core        WARNING 	Cannot load laps, telemetry, weather, and message data because the relevant API is not supported for this session.
core           INFO 	Finished loading data for 24 drivers: ['6', '5', '11', '7', '2', '8', '4', '14', '15', '16', '17', '3', '12', '20', '18', '19', '21', '9', '23', '25', '24', '22', '1', '10']
core           INFO 	Loading data for British Grand Prix - Race [v3.8.1]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...


Skip 2010 Monaco Grand Prix
Lade 2010 British Grand Prix...


logger      WARNING 	Failed to load session info data!
core        WARNING 	Cannot load laps, telemetry, weather, and message data because the relevant API is not supported for this session.
core           INFO 	Finished loading data for 24 drivers: ['6', '2', '4', '1', '9', '23', '5', '14', '3', '10', '15', '16', '12', '8', '7', '18', '19', '24', '20', '21', '17', '22', '11', '25']
core           INFO 	Loading data for Monaco Grand Prix - Race [v3.8.1]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...


Skip 2010 British Grand Prix
Lade 2011 Monaco Grand Prix...


...

Erfolg: 2024 British Grand Prix
Lade 2025 Monaco Grand Prix...


req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['4', '16', '81', '1', '44', '6', '31', '30', '23', '55', '63', '87', '43', '5', '18', '27', '22', '12', '14', '10']
core           INFO 	Loading data for British Grand Prix - Race [v3.8.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...


Erfolg: 2025 Monaco Grand Prix
Lade 2025 British Grand Prix...


req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['4', '81', '27', '44', '1', '10', '18', '23', '14', '63', '87', '55', '31', '16', '22', '12', '6', '5', '30', '43']


Erfolg: 2025 British Grand Prix


In [5]:
import fastf1
import sqlite3
import pandas as pd
import os

In [7]:
import sqlite3
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder

def get_ml_ready_data():
    conn = sqlite3.connect('f1_project.db')
    # Wir filtern Outlaps und Pitstops raus, da diese die Regression extrem verzerren
    query = "SELECT * FROM laptimes WHERE IsOutlap = 0 AND IsPitstop = 0"
    df = pd.read_sql(query, conn)
    conn.close()

    # Features (X) und Zielvariable (y)
    # Wir lassen Year und Track erst mal raus oder nutzen Track als One-Hot
    categorical_features = ['Team', 'Compound', 'Track']
    numerical_features = ['LapNumber', 'TyreLife', 'AirTemp', 'IsRaining']
    
    # One-Hot Encoding für die Kategorien
    df_encoded = pd.get_dummies(df, columns=categorical_features)
    
    # Zielvariable: LapTimeSec
    X = df_encoded.drop(['LapTimeSec', 'Year'], axis=1, errors='ignore')
    y = df_encoded['LapTimeSec']
    
    return train_test_split(X, y, test_size=0.2, random_state=42)

X_train, X_test, y_train, y_test = get_ml_ready_data()

In [8]:
from sklearn.tree import DecisionTreeRegressor
tree_model = DecisionTreeRegressor(max_depth=10)
tree_model.fit(X_train, y_train)

,criterion,'squared_error'
,splitter,'best'
,max_depth,10
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,None
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,ccp_alpha,0.0


In [9]:
from sklearn.ensemble import RandomForestRegressor
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

,n_estimators,100
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [10]:
import xgboost
from xgboost import XGBRegressor
xgb_model = XGBRegressor(n_estimators=100, learning_rate=0.1)
xgb_model.fit(X_train, y_train)

,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [11]:
from sklearn.metrics import mean_absolute_error

models = {"Tree": tree_model, "Forest": rf_model, "XGB": xgb_model}

for name, model in models.items():
    preds = model.predict(X_test)
    mae = mean_absolute_error(y_test, preds)
    print(f"{name} MAE: {mae:.3f} Sekunden")

Tree MAE: 1.768 Sekunden
Forest MAE: 1.539 Sekunden
XGB MAE: 1.737 Sekunden


In [12]:
import plotly.express as px

# Am Beispiel des Random Forest
importances = rf_model.feature_importances_
feature_names = X_train.columns
fig = px.bar(x=feature_names, y=importances, title="Was beeinflusst die Rundenzeit am meisten?")
fig.show()

In [51]:
import pandas as pd

def predict_custom_scenario(model, feature_columns):
    # Wir erstellen ein Szenario: 
    # Red Bull, Silverstone, Soft Reifen, Runde 10, 5 Runden auf dem Reifen, 25 Grad, Trocken
    scenario = {
        'LapNumber': 10,
        'TyreLife': 5,
        'AirTemp': 25.0,
        'IsRaining': 0,
        'IsPitstop': 1,
        'Team_Red Bull Racing': 1,
        'Track_Monaco Grand Prix': 1,
        'Compound_SOFT': 1
    }
    
    # In DataFrame umwandeln
    df_test = pd.DataFrame([scenario])
    
    # WICHTIG: Fehlende Spalten (andere Teams/Tracks) mit 0 auffüllen
    for col in feature_columns:
        if col not in df_test.columns:
            df_test[col] = 0
            
    # Spalten sortieren, damit sie exakt wie im Training sind
    df_test = df_test[feature_columns]
    
    prediction = model.predict(df_test)
    return prediction[0]

# Anwendung
# time = predict_custom_scenario(xgb_model, X_train.columns)
# print(f"Vorhergesagte Rundenzeit: {time:.3f} Sekunden")

In [53]:
time_xgb = predict_custom_scenario(xgb_model, X_train.columns)
print(f"Vorhergesagte Rundenzeit mit XGB: {time_xgb:.3f} Sekunden")

time_rf = predict_custom_scenario(rf_model, X_train.columns)
print(f"Vorhergesagte Rundenzeit mit Random Forest: {time_rf:.3f} Sekunden")

time_tree = predict_custom_scenario(tree_model, X_train.columns)
print(f"Vorhergesagte Rundenzeit mit Decision Tree: {time_tree:.3f} Sekunden")

Vorhergesagte Rundenzeit mit XGB: 78.266 Sekunden
Vorhergesagte Rundenzeit mit Random Forest: 78.541 Sekunden
Vorhergesagte Rundenzeit mit Decision Tree: 78.332 Sekunden


In [38]:
session = fastf1.get_session(2023, 'Monaco Grand Prix', 'R')
session.load(laps=True, telemetry=False, weather=True, messages=True)
test_lap = session.laps.pick_teams(['Red Bull Racing'])

core           INFO 	Loading data for Monaco Grand Prix - Race [v3.8.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['1', '14', '31', '44', '63', '16', '10', '55', '4', '81', '77', '21', '24', '23', '22', '11', '27', '2', '20', '18']


In [54]:
test_lap

,Time,Driver,DriverNumber,LapTime,LapNumber,Stint,PitOutTime,PitInTime,Sector1Time,Sector2Time,...,FreshTyre,Team,LapStartTime,LapStartDate,TrackStatus,Position,Deleted,DeletedReason,FastF1Generated,IsAccurate
0,0 days 01:03:27.435000,VER,1,0 days 00:01:24.238000,1.0,1.0,NaT,NaT,NaT,0 days 00:00:37.420000,...,True,Red Bull Racing,0 days 01:02:02.950000,NaT,12,1.0,False,,False,False
1,0 days 01:04:46.802000,VER,1,0 days 00:01:19.367000,2.0,1.0,NaT,NaT,0 days 00:00:20.954000,0 days 00:00:37.366000,...,True,Red Bull Racing,0 days 01:03:27.435000,NaT,1,1.0,False,,False,True
2,0 days 01:06:05.876000,VER,1,0 days 00:01:19.074000,3.0,1.0,NaT,NaT,0 days 00:00:20.854000,0 days 00:00:37.288000,...,True,Red Bull Racing,0 days 01:04:46.802000,NaT,1,1.0,False,,False,True
3,0 days 01:07:24.005000,VER,1,0 days 00:01:18.129000,4.0,1.0,NaT,NaT,0 days 00:00:20.835000,0 days 00:00:36.637000,...,True,Red Bull Racing,0 days 01:06:05.876000,NaT,1,1.0,False,,False,True
4,0 days 01:08:42.024000,VER,1,0 days 00:01:18.019000,5.0,1.0,NaT,NaT,0 days 00:00:20.745000,0 days 00:00:36.734000,...,True,Red Bull Racing,0 days 01:07:24.005000,NaT,1,1.0,False,,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
227,0 days 02:45:47.256000,PER,11,0 days 00:01:29.537000,72.0,6.0,NaT,NaT,0 days 00:00:23.628000,0 days 00:00:43.487000,...,True,Red Bull Racing,0 days 02:44:17.719000,NaT,1,17.0,False,,False,True
228,0 days 02:47:14.076000,PER,11,0 days 00:01:26.820000,73.0,6.0,NaT,NaT,0 days 00:00:22.267000,0 days 00:00:42.197000,...,True,Red Bull Racing,0 days 02:45:47.256000,NaT,1,17.0,False,,False,True
229,0 days 02:48:39.780000,PER,11,0 days 00:01:25.704000,74.0,6.0,NaT,NaT,0 days 00:00:22.036000,0 days 00:00:41.152000,...,True,Red Bull Racing,0 days 02:47:14.076000,NaT,1,17.0,False,,False,True
230,0 days 02:50:13.928000,PER,11,0 days 00:01:34.148000,75.0,6.0,NaT,NaT,0 days 00:00:23.990000,0 days 00:00:46.718000,...,True,Red Bull Racing,0 days 02:48:39.780000,NaT,1,17.0,False,,False,True
